# For each model config, we need to fix the results from incorrect denormalization

In [2]:
SMART_RESIZE = (1932, 1092)
IMAGE_SIZE = (1920, 1080)

W_RATIO = IMAGE_SIZE[0] / SMART_RESIZE[0]
H_RATIO = IMAGE_SIZE[1] / SMART_RESIZE[1]

RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/results_baseline copy"

# folder structure for RESULT_FOLDER_PATH:
# results_baseline/
#   locke_<model_name>_<reasoning_config>_<task_type>/
#     run_<timestamp>_test_<split_name>_<variant>/
#       uitars_predictions/
#         <task_id>.jsonl
#       uitars_metrics.jsonl
#       uitars_summary.jsonl

# Fixing the predicted action coordinates

In [3]:
# For each model config folder, for each run folder, 
# in its uitars_predictions.jsonl, find the "prediction" which has 
# "Action: click(start_box='(390,546)')"
# and "predicted_actions" which has field like ["click(start_box='(390,546)')"]
# parse the coordinates and multiply with W_RATIO and H_RATIO and replace the original coordinates

import json
import re
import os
from pathlib import Path

def fix_coordinates_in_string(text, w_ratio, h_ratio):
    """
    Fix coordinates in strings like "click(start_box='(390,546)')"
    Returns the string with updated coordinates
    """
    def replace_coords(match):
        coords_str = match.group(1)
        # Parse coordinates
        x, y = map(int, coords_str.strip('()').split(','))
        # Apply ratios
        new_x = int(x * w_ratio)
        new_y = int(y * h_ratio)
        return f"({new_x},{new_y})"
    
    # Pattern to match coordinates in parentheses: (x,y)
    pattern = r'\((\d+,\d+)\)'
    return re.sub(pattern, replace_coords, text)

def fix_jsonl_file(file_path, w_ratio, h_ratio):
    """
    Fix coordinates in a single JSONL file
    """
    fixed_lines = []
    modified = False
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                fixed_lines.append(line)
                continue
                
            try:
                data = json.loads(line)
                line_modified = False
                
                # Fix prediction field
                if 'prediction' in data and isinstance(data['prediction'], str):
                    original = data['prediction']
                    fixed = fix_coordinates_in_string(original, w_ratio, h_ratio)
                    if fixed != original:
                        data['prediction'] = fixed
                        line_modified = True
                
                # Fix predicted_actions field
                if 'predicted_actions' in data and isinstance(data['predicted_actions'], list):
                    fixed_actions = []
                    for action in data['predicted_actions']:
                        if isinstance(action, str):
                            original_action = action
                            fixed_action = fix_coordinates_in_string(original_action, w_ratio, h_ratio)
                            fixed_actions.append(fixed_action)
                            if fixed_action != original_action:
                                line_modified = True
                        else:
                            fixed_actions.append(action)
                    data['predicted_actions'] = fixed_actions
                
                if line_modified:
                    modified = True
                    fixed_lines.append(json.dumps(data) + '\n')
                else:
                    fixed_lines.append(line)
                    
            except json.JSONDecodeError as e:
                print(f"Warning: Could not parse JSON in {file_path}: {e}")
                fixed_lines.append(line)
    
    # Write back if modified
    if modified:
        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(fixed_lines)
        return True
    return False

# Process all JSONL files in uitars_predictions folders
result_path = Path(RESULT_FOLDER_PATH)
total_files = 0
modified_files = 0

for model_config_folder in result_path.iterdir():
    if not model_config_folder.is_dir():
        continue
    
    print(f"Processing model config: {model_config_folder.name}")
    
    for run_folder in model_config_folder.iterdir():
        if not run_folder.is_dir():
            continue
        
        uitars_predictions_folder = run_folder / "uitars_predictions"
        if not uitars_predictions_folder.exists():
            continue
        
        print(f"  Processing run: {run_folder.name}")
        
        # Process all JSONL files in uitars_predictions
        for jsonl_file in uitars_predictions_folder.glob("*.jsonl"):
            total_files += 1
            if fix_jsonl_file(jsonl_file, W_RATIO, H_RATIO):
                modified_files += 1
                print(f"    Fixed: {jsonl_file.name}")

print(f"\nSummary: Processed {total_files} files, modified {modified_files} files")


Processing model config: locke_qwen_NO_thought_relational_query
  Processing run: run_20251113_005227_test_task_text_shrink
    Fixed: 8dcf6423-262a-439b-9ee7-279a920468fa.jsonl
    Fixed: e6f6e6c8-f1e6-42bb-a3af-696ed8de571b.jsonl
    Fixed: 400c291f-6a0c-46fb-874e-d5c174fdedfc.jsonl
    Fixed: 5b888855-b921-4c61-8f79-73902ee0eafa.jsonl
    Fixed: 9ceab2a3-7919-4f15-871a-21638fd93b24.jsonl
    Fixed: 3c76cc80-ddcb-48ed-941d-74cffecfc33f.jsonl
    Fixed: 2cc71f04-851c-4a75-8728-a80783984a32.jsonl
    Fixed: 3c10b271-ae24-4289-9504-e36ab8565243.jsonl
    Fixed: 790ba0ec-4e7d-4df0-ac86-ea52b3a73532.jsonl
    Fixed: 19b955ba-fdcd-4345-b33a-fc6a88b5a85d.jsonl
    Fixed: dd8a2207-a5b0-4116-a63d-b62835d68b4e.jsonl
    Fixed: 18fc60d7-aa69-4c07-9bf1-64543eae52c9.jsonl
    Fixed: 14d50319-3f81-4aa6-8ee8-d1b66e4d5d64.jsonl
    Fixed: ed60077a-1853-4b0d-8174-b339d08de32e.jsonl
    Fixed: 5c29c805-388d-471a-80e9-ca0fbaf820be.jsonl
    Fixed: 91695df8-f256-47c9-8c37-06e8d0fc758f.jsonl
    Fixed: 3

# Fixing the metric calculation in uitars_metrics.jsonl and uitars_summary.jsonl

In [ ]:
# Now we need to fix the downstream metrics calculation
# We have 3 metrics:
# action_str_em (exact match)
# hit_box_accuracy
# bbox_center_mse
# and the logic for calculating them are:
from typing import Optional, Any

def uitars_action_to_mind2web_op(action_type: str, action_inputs: dict[str, str]) -> Optional[str]:
    """
    Map UITARS action_type to Mind2Web 'op' string.
    
    Args:
        action_type: UITARS action name (e.g., 'click', 'type', 'scroll')
        action_inputs: Dictionary of action parameters (e.g., {'content': 'hello'})
    
    Returns:
        Mind2Web operation code (e.g., 'CLICK', 'TYPE', 'SCROLL')
    
    Examples:
        >>> uitars_action_to_mind2web_op('click', {})
        'CLICK'
        >>> uitars_action_to_mind2web_op('type', {'content': '\\n'})
        'PRESS ENTER'
        >>> uitars_action_to_mind2web_op('hotkey', {'key': 'enter'})
        'HOTKEY'
    """
    action_type = (action_type or "").lower().strip()
    
    # Direct mappings
    ACTION_TYPE_TO_OP = {
        "click": "CLICK",
        "left_double": "CLICK",  # Mind2Web doesn't distinguish double-click, map to CLICK
        "right_single": "CLICK",  # Mind2Web doesn't have right-click, map to CLICK
        "drag": "DRAG",
        "scroll": "SCROLL",
        "wait": "IGNORE",
        "finished": "IGNORE",  # No direct Mind2Web equivalent
    }
    
    # Check for direct mapping
    if action_type in ACTION_TYPE_TO_OP:
        return ACTION_TYPE_TO_OP[action_type]
    
    # Special handling for type action
    if action_type == "type":
        content = action_inputs.get("content", "")
        # Check if it's a submit action (ends with newline)
        if content and content.strip() == "\\n":
            return "ENTER"
        return "TYPE"
    
    # Special handling for hotkey
    if action_type == "hotkey":
        key = action_inputs.get("key", "").lower()
        # Map common hotkeys to specific Mind2Web ops
        if key in ("enter", "return", "\\n"):
            return "ENTER"
        return "HOTKEY"
    
    return None


def get_action_str_em(predicted_action, predicted_action_inputs, gt_action, gt_action_inputs: Optional[Any]):
    """
    Args:
        predicted_action: UITARS action name (e.g., 'click', 'type', 'scroll')
        predicted_action_inputs: Dictionary of action parameters (e.g., {'content': 'hello'})
        gt_action: "op" from <task_id>.jsonl (e.g., 'CLICK', 'TYPE', 'SCROLL')
        gt_action_inputs: Dictionary of action parameters (e.g., {'content': 'hello'})
    """
    mind2web_action = uitars_action_to_mind2web_op(predicted_action['action_type'], predicted_action_inputs)
    return 1 if mind2web_action == gt_action.lower() else 0


def get_hit_box_accuracy(predicted_point, gt_bbox):
    x, y = predicted_point
    if len(gt_bbox) < 4:
        return False
    bx, by, bw, bh = map(float, gt_bbox[:4])
    return 1 if bx <= x <= bx + bw and by <= y <= by + bh else 0


def get_bbox_center_mse(predicted_point, gt_bbox):
    dx = predicted_point[0] - gt_bbox[0]
    dy = predicted_point[1] - gt_bbox[1]
    return dx * dx + dy * dy



# Fix total metrics and summary per variant